In [5]:

from cipo import user_copy

ImportError: cannot import name 'get_observatory_location' from 'cipo.user_copy' (C:\Users\lucas\OneDrive\Documentos\acai\cipo\src\cipo\user_copy.py)

In [ ]:
# Executa a coleta dinâmica, aplica o filtro de visibilidade e gera o gráfico
df_visiveis = user_copy.analyze_ephemeris_objects(
    obs_code='Y28', 
    obj_type='NEOCP', 
    altitude_min=20, 
    duration_min=45, 
    plot=True
)

# Exibe a tabela final com os melhores alvos
display(df_visiveis)

In [ ]:
df_complete = user.mpc_objects(type_object)

In [ ]:
df_complete

In [ ]:

from cipo.main import analyze_ephemeris_objects
df_resultado = analyze_ephemeris_objects(obj_type='NEOCP', plot=True)

if df_resultado is not None and not df_resultado.empty:
    display(df_resultado[['Temp Desig', 'R.A.', 'Decl.', 'V', 'Visible_Minutes', 'Max_Alt']])
else:
    print("\nNenhum objeto visível encontrado.")

In [ ]:
# %% [markdown]
# ## Análise completa: Preview + Efemérides com cruzamento

# %%
# 1. Defina seus parâmetros
code_obs = 'Y28'          # Código do observatório
type_object = 'NEOCP'     # 'NEOCP' ou 'PCCP'
altitude_min = 20         # Altitude mínima (graus)
time_min = 30             # Duração mínima contínua (minutos)

# 2. Preview Estático (Tabela 1 - Visão Geral do Site)
print("1. Obtendo preview da página inicial...")
df_preview = user_copy._download_mpc_table(type_object)   # Agora com user_copy.

if df_preview is None:
    print("Erro ao baixar preview. Verifique a conexão.")
else:
    # Exibe as primeiras linhas para confirmar
    print(f"Preview carregado: {len(df_preview)} objetos.")
    display(df_preview[['Temp Desig', 'Score', 'V', 'Updated']].head())

    # 3. Coleta Dinâmica (Tabela 2 - Efemérides via Selenium)
    print("\n2. Baixando efemérides detalhadas para o observatório", code_obs)
    texto_bruto = user_copy.fetch_mpc_data(type_object, code_obs)

    if texto_bruto:
        # Parse do texto para dicionário de DataFrames
        dict_efemerides = user_copy.parse_mpc_data(texto_bruto)
        print(f"Efemérides obtidas para {len(dict_efemerides)} objetos.")

        # 4. Filtro de visibilidade (usa Object Alt e Sun Alt)
        df_filtrado = user_copy.filter_visible_objects(
            dict_efemerides,
            altitude_min=altitude_min,
            time_min_minutes=time_min
        )

        if not df_filtrado.empty:
            # Ordena por altitude máxima (melhor alvo primeiro)
            df_ordenado = df_filtrado.sort_values('Max_Alt', ascending=False)
            object_selected = df_ordenado.iloc[0]['Temp Desig']

            print(f"\n3. Sucesso! {len(df_filtrado)} alvos viáveis encontrados.")
            print(f"Melhor alvo para apontamento: {object_selected}")

            # 5. Cruzamento: adiciona Score e Updated da tabela de preview
            resultado_final = df_ordenado.merge(
                df_preview[['Temp Desig', 'Score', 'Updated']],
                on='Temp Desig',
                how='left'
            )

            # Exibe colunas mais relevantes
            cols = ['Temp Desig', 'R.A.', 'Decl.', 'V', 'Score', 'Visible_Minutes', 'Max_Alt', 'Max_Alt_Time_UTC', 'Updated']
            display(resultado_final[cols])

            # (Opcional) Plotar curvas de altitude para todos os visíveis
            plotar = input("Deseja plotar as curvas de altitude? (s/n): ").strip().lower()
            if plotar == 's':
                import matplotlib.pyplot as plt
                from astropy.time import Time
                import pandas as pd
                plt.figure(figsize=(12, 6))
                now_dt = pd.to_datetime(Time.now().datetime)
                for obj in resultado_final['Temp Desig']:
                    df_obj = dict_efemerides[obj].copy()
                    ut_clean = df_obj['UT'].astype(str).str.replace(r'[\s\.]', '', regex=True).str.zfill(4)
                    df_obj['Datetime_UTC'] = pd.to_datetime(
                        df_obj['Date'] + ' ' + ut_clean,
                        format='%Y %m %d %H%M',
                        errors='coerce'
                    )
                    df_obj = df_obj.dropna(subset=['Datetime_UTC'])
                    df_obj['Object Alt'] = pd.to_numeric(df_obj['Object Alt'], errors='coerce')
                    times_hours = (df_obj['Datetime_UTC'] - now_dt).dt.total_seconds() / 3600
                    plt.plot(times_hours, df_obj['Object Alt'], label=obj, marker='.', linestyle='-', linewidth=1)

                plt.axhline(altitude_min, color='red', linestyle='--', label=f'Limite {altitude_min}°')
                plt.ylim(0, 90)
                plt.xlabel('Horas a partir de agora (UTC)')
                plt.ylabel('Altitude (°)')
                plt.title(f'Curvas de altitude para objetos visíveis (obs. {code_obs})')
                plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

        else:
            print("\n3. Nenhum objeto do MPC atinge os critérios de visibilidade hoje.")
    else:
        print("\nFalha ao obter os dados profundos do Minor Planet Center.")

In [ ]:
texto = user_copy.fetch_mpc_data(type_object, code_obs)
print(texto[:2000])  # veja os primeiros 2000 caracteres


In [ ]:
df_resultado = user.analyze_ephemeris_objects(code_obs, type_object, altitude_min, time_min, plot=True)

if df_resultado is not None and not df_resultado.empty:
    display(df_resultado[['Temp Desig', 'R.A.', 'Decl.', 'V', 'Visible_Minutes', 'Max_Alt']])
else:
    print("\nNenhum objeto visível encontrado.")

In [ ]:
df_visible = user.process_mpc_data(code_obs, type_object, interactive_mode=True)

In [ ]:

import importlib
importlib.reload(user) # Força o Python a ler o arquivo user.py atualizado!

# Roda a função
resultado = user.analyze_ephemeris_objects(code_obs, type_object, altitude_min, time_min, plot=True)

# Trava de segurança dupla
if resultado is not None and not resultado['visible_objects'].empty:
    display(resultado['visible_objects'][['Temp Desig', 'R.A.', 'Decl.', 'V', 'Visible_Minutes', 'Max_Alt']])
else:
    print("\nNenhum objeto visível encontrado ou falha na obtenção dos dados.")